# NIH ChestX-ray14 — 14-Class Training on Kaggle
### EfficientNetB3 · Patient-Level Split · Mean AUC Target: 0.82

**Before running any cells:**
1. ✅ Add the NIH dataset: right sidebar → **Add data** → search `nih-chest-xrays/data` → Add
2. ✅ Add your project code dataset: right sidebar → **Add data** → search your dataset name → Add
3. ✅ Enable GPU: right sidebar → **Session options → Accelerator → GPU T4 x2**
4. ✅ Enable internet: right sidebar → **Session options → Internet → On**
5. ✅ Add W&B secret (see Cell 4)


## Cell 1 — Confirm GPU

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:800])


Sat Jun 27 11:42:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10


## Cell 2 — Extract project code

Your project zip must be added as a Kaggle dataset first.
See the setup guide for how to do that (it takes 2 minutes).


In [2]:
import zipfile, os, sys

# --- EDIT THIS: set the name of YOUR Kaggle dataset ---
# After adding your dataset, check the path in /kaggle/input/
# It will be something like: /kaggle/input/chest-xray-code/
DATASET_PATH = '/kaggle/input/datasets/rishabhgalave/chest-xray-code/chest-xray-nih-extension'   # ← change if your dataset slug differs

# Find the zip inside your dataset
# zip_files = [f for f in os.listdir(DATASET_PATH) if f.endswith('.zip')]
# if not zip_files:
#     raise FileNotFoundError(
#         f"No .zip found in {DATASET_PATH}\n"
#         f"Files present: {os.listdir(DATASET_PATH)}"
#     )

# zip_path = os.path.join(DATASET_PATH, zip_files[0])
# print(f"Extracting: {zip_path}")

# with zipfile.ZipFile(zip_path, 'r') as zf:
#     zf.extractall('/kaggle/working/')

# Find the project root (the folder containing src/ and run_training_nih.py)
PROJECT_DIR = DATASET_PATH
# for root, dirs, files in os.walk('/kaggle/working'):
#     if 'src' in dirs and 'run_training_nih.py' in files:
#         PROJECT_DIR = root
#         break

# if PROJECT_DIR is None:
#     raise RuntimeError(
#         "Could not find project root. "
#         "Make sure chest-xray-detector-complete.zip was uploaded correctly."
#     )

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Project directory: {PROJECT_DIR}")
print("Contents:", [f for f in os.listdir(PROJECT_DIR) if not f.startswith('.')])


Project directory: /kaggle/input/datasets/rishabhgalave/chest-xray-code/chest-xray-nih-extension
Contents: ['tests', 'run_evaluation_nih.py', 'README.md', 'config.YAML', 'requirements.txt', 'run_training_nih.py', 'notebooks', 'src']


## Cell 3 — Install extra dependencies

In [3]:
# Kaggle pre-installs: torch, torchvision, sklearn, pandas, numpy, PIL
# We only need to add the packages Kaggle doesn't have by default
# !pip install -q wandb==0.17.3 huggingface-hub==0.23.4

# Verify key imports
import torch, torchvision, sklearn, wandb
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"sklearn     : {sklearn.__version__}")
print(f"wandb       : {wandb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


torch       : 2.10.0+cu128
torchvision : 0.25.0+cu128
sklearn     : 1.6.1
wandb       : 0.25.1
CUDA available: True
GPU: Tesla T4


## Cell 4 — Connect Weights & Biases via Kaggle Secrets

**One-time setup (do this before running the notebook):**
1. Go to kaggle.com → your profile picture → **Settings**
2. Scroll to **API** section → click **Add new secret** (or use Kaggle Secrets)
3. Or: in the notebook right sidebar → **Add-ons → Secrets → Add a new secret**
4. Name: `WANDB_API_KEY`  |  Value: your key from [wandb.ai/authorize](https://wandb.ai/authorize)

Skip this cell and pass `--no-wandb` to training commands if you don't want W&B.


In [4]:
USE_WANDB = True

try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("WANDB_API_KEY")
    import wandb
    wandb.login(key=wandb_key, relogin=True)
    print("W&B login successful.")
except Exception as e:
    print(f"W&B login skipped: {e}")
    print("Training will run without experiment tracking. Pass --no-wandb below.")
    USE_WANDB = False


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rishabhgalave (rishabhgalave-rishabhgalave) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


## Cell 5 — Locate and verify the NIH dataset

The NIH dataset is already available at `/kaggle/input/data/` — no download needed.
This cell confirms the structure is correct before any training starts.


In [5]:
import os, glob

# Standard Kaggle path for nih-chest-xrays/data
DATA_ROOT = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'

print("Checking dataset structure...")
print(f"DATA_ROOT: {DATA_ROOT}")
print()

required = [
    'images_001/images',
    'images_002/images',
    'images_003/images',
    'images_004/images',
    'images_005/images',
    'images_006/images',
    'images_007/images',
    'images_008/images',
    'images_009/images',
    'images_010/images',
    'images_011/images',
    'images_012/images',
    'Data_Entry_2017.csv',
    'train_val_list.txt',
    'test_list.txt',
]
images = [
    'images_001/images',
    'images_002/images',
    'images_003/images',
    'images_004/images',
    'images_005/images',
    'images_006/images',
    'images_007/images',
    'images_008/images',
    'images_009/images',
    'images_010/images',
    'images_011/images',
    'images_012/images',
]

all_ok = True
for item in required:
    path = os.path.join(DATA_ROOT, item)
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {item}")
    if not exists:
        all_ok = False

print()
if all_ok:
    # Count images
    png_count = 0
    for items in images:
        png_count += len(glob.glob(f"{DATA_ROOT}/{items}/*.png"))
    print(f"PNG files in images/: {png_count:,}")
    if png_count >= 112000:
        print("✓  Full dataset present (112,120 images)")
    else:
        print(f"⚠  Only {png_count:,} images found — expected 112,120")
        print("   Try removing and re-adding the dataset in the sidebar.")

    # Check CSV
    import pandas as pd
    df = pd.read_csv(f"{DATA_ROOT}/Data_Entry_2017.csv")
    print(f"CSV rows: {len(df):,}")

    # Check split files
    train_val = open(f"{DATA_ROOT}/train_val_list.txt").read().splitlines()
    test_imgs = open(f"{DATA_ROOT}/test_list.txt").read().splitlines()
    print(f"train_val_list.txt: {len(train_val):,} entries")
    print(f"test_list.txt:      {len(test_imgs):,} entries")
else:
    print("\n✗  Dataset structure incomplete. Re-add the NIH dataset from the sidebar.")


Checking dataset structure...
DATA_ROOT: /kaggle/input/datasets/organizations/nih-chest-xrays/data

  ✓  images_001/images
  ✓  images_002/images
  ✓  images_003/images
  ✓  images_004/images
  ✓  images_005/images
  ✓  images_006/images
  ✓  images_007/images
  ✓  images_008/images
  ✓  images_009/images
  ✓  images_010/images
  ✓  images_011/images
  ✓  images_012/images
  ✓  Data_Entry_2017.csv
  ✓  train_val_list.txt
  ✓  test_list.txt

PNG files in images/: 112,120
✓  Full dataset present (112,120 images)
CSV rows: 112,120
train_val_list.txt: 86,524 entries
test_list.txt:      25,596 entries


## Cell 6 — Configure DATA_ROOT and checkpoint directory

Checkpoints are saved to `/kaggle/working/checkpoints/`.
After training, download them from the **Output** tab on the right sidebar.


In [6]:
import os
from src.config_nih import CONFIG

# Kaggle-specific paths
DATA_ROOT     = '/kaggle/input/datasets/organizations/nih-chest-xrays/data'
CHECKPOINT_DIR = '/kaggle/working/checkpoints/nih_chestxray'

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Apply to config
CONFIG.DATA_ROOT      = DATA_ROOT
CONFIG.CHECKPOINT_DIR = CHECKPOINT_DIR
CONFIG.USE_WANDB      = USE_WANDB

print(f"DATA_ROOT      : {CONFIG.DATA_ROOT}")
print(f"CHECKPOINT_DIR : {CONFIG.CHECKPOINT_DIR}")
print(f"NUM_CLASSES    : {CONFIG.NUM_CLASSES}")
print(f"BATCH_SIZE     : {CONFIG.BATCH_SIZE}")
print(f"USE_WANDB      : {CONFIG.USE_WANDB}")
print()
print("14 classes:")
for i, cls in enumerate(CONFIG.CLASS_NAMES):
    print(f"  [{i:2d}] {cls}")


DATA_ROOT      : /kaggle/input/datasets/organizations/nih-chest-xrays/data
CHECKPOINT_DIR : /kaggle/working/checkpoints/nih_chestxray
NUM_CLASSES    : 14
BATCH_SIZE     : 32
USE_WANDB      : True

14 classes:
  [ 0] Atelectasis
  [ 1] Cardiomegaly
  [ 2] Effusion
  [ 3] Infiltration
  [ 4] Mass
  [ 5] Nodule
  [ 6] Pneumonia
  [ 7] Pneumothorax
  [ 8] Consolidation
  [ 9] Edema
  [10] Emphysema
  [11] Fibrosis
  [12] Pleural_Thickening
  [13] Hernia


## Cell 7 — Run the NIH test suite

In [7]:
!python -m pytest tests/test_dataset_nih.py -v --tb=short 2>&1 | tail -40


tests/test_dataset_nih.py::TestNIHChestXrayDataset::test_get_labels_matrix_shape PASSED [ 87%]
tests/test_dataset_nih.py::TestNIHDataLoaders::test_keys PASSED          [ 90%]
tests/test_dataset_nih.py::TestNIHDataLoaders::test_train_image_batch_shape PASSED [ 93%]
tests/test_dataset_nih.py::TestNIHDataLoaders::test_train_label_batch_shape PASSED [ 96%]
tests/test_dataset_nih.py::TestNIHDataLoaders::test_pos_weight_shape PASSED [100%]

=============================== warnings summary ===============================
tests/test_dataset_nih.py::TestNIHDataLoaders::test_keys
  /usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/free

## Cell 8 — Smoke test (1 epoch, no pretrained weights, no W&B)

Takes ~5 minutes. Confirms the full 14-class pipeline runs without errors
before spending 6+ hours on the real training run.


In [ ]:
no_wandb_flag = "" if USE_WANDB else "--no-wandb"

!python run_training_nih.py \
    --data-root   {DATA_ROOT} \
    --checkpoint-dir /kaggle/working/checkpoints/smoketest \
    --epochs1 1 \
    --epochs2 0 \
    --no-pretrained \
    {no_wandb_flag}


## Cell 9 — Phase 1: Train head only (5 epochs, backbone frozen)

**~25 min/epoch × 5 epochs ≈ 2 hours**

What to expect:
- Epoch 1: mean AUC ~0.60–0.65 (head learning from scratch)
- Epoch 3: mean AUC ~0.68–0.72
- Epoch 5: mean AUC ~0.72–0.76
- Gate to clear: **mean val AUC > 0.70**

A per-class AUC table prints after every epoch so you can monitor progress.


In [ ]:
no_wandb_flag = "" if USE_WANDB else "--no-wandb"

!python run_training_nih.py \
    --data-root      {DATA_ROOT} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs1 5 \
    --epochs2 0 \
    {no_wandb_flag}


## Cell 10 — Check Phase 1 gate (must clear before Phase 2)

In [ ]:
import torch

ckpt_path = f"{CHECKPOINT_DIR}/best_model.pth"
ckpt = torch.load(ckpt_path, map_location='cpu')

print(f"Best epoch    : {ckpt['epoch'] + 1}")
print(f"Best mean AUC : {ckpt['best_auc']:.4f}")
print()

if ckpt['best_auc'] > 0.70:
    print("✓  PHASE 1 GATE CLEARED")
    print("   Proceed to Cell 11 for Phase 2 fine-tuning.")
else:
    print("✗  Gate not cleared.")
    print()
    print("   Possible causes:")
    print("   1. Model downloaded ImageNet weights? → check Cell 9 output for 'pretrained=True'")
    print("   2. pos_weight on wrong device? → should see pos_weight shape [14] in logs")
    print("   3. Loss not decreasing? → check train/loss column in epoch logs")
    print()
    print("   Fix: Ensure --no-pretrained is NOT in the Cell 9 command, then re-run Cell 9.")


## Cell 11 — Phase 2: Fine-tune top 3 MBConv blocks (10 epochs)

**~27 min/epoch × 10 epochs ≈ 4.5 hours**

This re-runs Phase 1 (fast — ~2 hours) then continues to Phase 2.
Total wall-clock: ~6.5 hours. The Kaggle session limit is 12 hours — you have margin.

What to expect in Phase 2:
- Epoch 1: mean AUC jumps +3–4 pts from Phase 1 end
- Epochs 2–5: steady improvement to ~0.83–0.85
- Epochs 6–10: plateau or small gains; early stopping may trigger

The best checkpoint (highest mean val AUC) is saved automatically.


In [8]:
no_wandb_flag = "" if USE_WANDB else "--no-wandb"

!python run_training_nih.py \
    --data-root      {DATA_ROOT} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --epochs1 5 \
    --epochs2 10 \
    {no_wandb_flag}


11:43:36 | INFO     | src.utils            | All random seeds set to 42
11:43:36 | INFO     | src.utils            | Device: Tesla T4 | VRAM: 15.6 GB | CUDA: 12.8
11:43:36 | INFO     | __main__             | Loading NIH ChestX-ray14 dataset...
11:43:37 | INFO     | src.dataset_nih      | Indexed 112120 images across 12 images_*/images folders
11:46:30 | INFO     | src.dataset_nih      | CSV loaded: 112120 entries from Data_Entry_2017.csv
11:46:30 | INFO     | src.dataset_nih      | Official splits — train_val: 86524 images | test: 25596 images
11:46:30 | INFO     | src.dataset_nih      |   train:  73916 images | 23806 patients
11:46:30 | INFO     | src.dataset_nih      |   val  :  12608 images |  4202 patients
11:46:30 | INFO     | src.dataset_nih      |   test :  25596 images |  2797 patients
11:46:30 | INFO     | src.dataset_nih      |   Atelectasis         : pos=  7003  neg= 66913  pos_weight=9.55
11:46:30 | INFO     | src.dataset_nih      |   Cardiomegaly        : pos=  1453  neg= 

## Cell 12 — Final gate check and per-class summary

In [10]:
import torch, json
import numpy as np

ckpt = torch.load(f"{CHECKPOINT_DIR}/best_model.pth", map_location='cpu', weights_only=False)
mean_auc = ckpt['best_auc']

print(f"{'='*52}")
print(f"  TRAINING COMPLETE")
print(f"{'='*52}")
print(f"  Best val mean AUC : {mean_auc:.4f}")
print(f"  CheXNet benchmark : 0.8427")
delta = mean_auc - 0.8427
arrow = "↑" if delta >= 0 else "↓"
print(f"  Δ vs CheXNet      : {arrow}{abs(delta):.4f}")
print(f"{'='*52}")
print()

if mean_auc >= 0.82:
    print("✓  NIH GATE CLEARED — ready for test-set evaluation")
    print()
    print("  Next steps:")
    print("  1. Download best_model.pth from the Output tab (right sidebar)")
    print("  2. Save as: checkpoints/nih_best_model.pth on your local machine")
    print("  3. Run: python run_evaluation_nih.py --checkpoint checkpoints/nih_best_model.pth")
else:
    print(f"✗  Mean AUC {mean_auc:.4f} < 0.82")
    print("   Try: --loss focal in Cell 11, or increase --epochs2 to 15")


  TRAINING COMPLETE
  Best val mean AUC : 0.8223
  CheXNet benchmark : 0.8427
  Δ vs CheXNet      : ↓0.0204

✓  NIH GATE CLEARED — ready for test-set evaluation

  Next steps:
  1. Download best_model.pth from the Output tab (right sidebar)
  2. Save as: checkpoints/nih_best_model.pth on your local machine
  3. Run: python run_evaluation_nih.py --checkpoint checkpoints/nih_best_model.pth


## Cell 13 — (Optional) Run in-notebook evaluation on test set

If you don't want to download the checkpoint and run evaluation locally,
you can run it here. Takes ~15 minutes on GPU (much faster than CPU).
Saves plots and JSON to `/kaggle/working/assets/nih/`.


In [14]:
import os
os.makedirs('/kaggle/working/assets/nih', exist_ok=True)

!python run_evaluation_nih.py \
    --checkpoint {CHECKPOINT_DIR}/best_model.pth \
    --data-root  {DATA_ROOT} \
    --assets-dir /kaggle/working/assets/nih

# The Output tab will show the generated PNG files for direct download
print("\nGenerated files:")
for f in os.listdir('/kaggle/working/assets/nih'):
    size_kb = os.path.getsize(f'/kaggle/working/assets/nih/{f}') // 1024
    print(f"  {f}  ({size_kb} KB)")


15:32:30 | INFO     | src.utils            | Device: Tesla T4 | VRAM: 15.6 GB | CUDA: 12.8
15:32:30 | INFO     | src.model            | Built EfficientNetB3 | pretrained=False | num_classes=14 | dropout=0.4 | head: 1536 → 14
15:32:31 | INFO     | src.utils            | Loaded checkpoint from /kaggle/working/checkpoints/nih_chestxray/best_model.pth (epoch=10, AUC=0.8223)
15:32:31 | INFO     | __main__             | Checkpoint: epoch=10, best val AUC=0.8223
15:32:33 | INFO     | src.dataset_nih      | Indexed 112120 images across 12 images_*/images folders
15:34:12 | INFO     | src.dataset_nih      | CSV loaded: 112120 entries from Data_Entry_2017.csv
15:34:12 | INFO     | src.dataset_nih      | Official splits — train_val: 86524 images | test: 25596 images
15:34:12 | INFO     | src.dataset_nih      |   train:  73916 images | 23806 patients
15:34:12 | INFO     | src.dataset_nih      |   val  :  12608 images |  4202 patients
15:34:12 | INFO     | src.dataset_nih      |   test :  25596 ima

## Cell 14 — Verify output files for download

After running Cell 12 and/or Cell 13, these files are available in the
**Output** tab (right sidebar) for download:
- `checkpoints/nih_chestxray/best_model.pth`  ← the trained model
- `checkpoints/nih_chestxray/last_model.pth`  ← latest epoch
- `assets/nih/roc_curves_14class.png`          ← 14-line ROC chart
- `assets/nih/auc_comparison_chexnet.png`      ← bar chart vs CheXNet
- `assets/nih/evaluation_results_nih.json`     ← metrics JSON


In [12]:
import os

output_dirs = [
    f'{CHECKPOINT_DIR}',
    '/kaggle/working/assets/nih',
]

print("Files available for download:")
print()
for d in output_dirs:
    if os.path.exists(d):
        print(f"{d}/")
        for f in sorted(os.listdir(d)):
            size = os.path.getsize(os.path.join(d, f))
            if size > 1024*1024:
                size_str = f"{size/1024/1024:.1f} MB"
            elif size > 1024:
                size_str = f"{size/1024:.1f} KB"
            else:
                size_str = f"{size} B"
            print(f"    {f}  ({size_str})")
        print()


Files available for download:

/kaggle/working/checkpoints/nih_chestxray/
    best_model.pth  (117.2 MB)
    last_model.pth  (117.2 MB)

/kaggle/working/assets/nih/



In [13]:
import os
import zipfile
from pathlib import Path

# ── Config ──────────────────────────────────────────────
OUTPUT_DIR  = "/kaggle/working"
ZIP_NAME    = "all_outputs.zip"
ZIP_PATH    = f"/kaggle/working/{ZIP_NAME}"

# ── Zip all files ────────────────────────────────────────
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file in Path(OUTPUT_DIR).rglob("*"):
        if file.is_file() and file.name != ZIP_NAME:  # skip the zip itself
            arcname = file.relative_to(OUTPUT_DIR)
            zipf.write(file, arcname)
            print(f"  + {arcname}")

# ── Summary ──────────────────────────────────────────────
size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"\n✅ Done → {ZIP_NAME}  ({size_mb:.1f} MB)")
print("📥 Go to Kaggle Output panel → click the file to download")

  + .virtual_documents/__notebook_source__.ipynb
  + checkpoints/nih_chestxray/best_model.pth
  + checkpoints/nih_chestxray/last_model.pth
  + checkpoints/smoketest/best_model.pth
  + checkpoints/smoketest/last_model.pth

✅ Done → all_outputs.zip  (292.3 MB)
📥 Go to Kaggle Output panel → click the file to download
